In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import MNIST

from sklearn.metrics import accuracy_score, classification_report

# Load & Preprocess

In [2]:

transform = transforms.Compose([
    # data augmentation
    transforms.RandomHorizontalFlip(),
    transforms.RandomResizedCrop(size=28, scale=(0.8, 1.0), antialias=True),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, hue=0.1),

    # conversion to tensor and normalization
    transforms.ToTensor(),
    transforms.Normalize((0.5), (0.5))
])


In [12]:
# Load dataset
train_set = MNIST(
    "datasets/mnist/",
    transform=transform,
    train=True,
    # True for first time download
    download=False,
)

test_set = MNIST(
    "datasets/mnist/",
    transform=transform,
    train=False,
)

In [4]:
# split train set to train & validation set
train_set, val_set = random_split(
    train_set,
    lengths=(0.8, 0.2)
)

In [13]:
# to dataloader
train_loader = DataLoader(
    train_set,
    shuffle=True,
    batch_size=64
)
val_loader = DataLoader(
    val_set, 
    shuffle=True,
    batch_size=64
)
test_loader = DataLoader(
    test_set,
    shuffle=True,
    batch_size=64
)

# CNN

## Define the model

In [6]:
# Image dimension: 1 x 28 x 28

class CNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.convo_layers = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, padding=1), # 8 x 28 x 28
            nn.ReLU(),

            nn.Conv2d(8, 8, kernel_size=3, padding=1), # 8 x 28 x 28
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # 8 x 14 x 14

            nn.Conv2d(8, 16, kernel_size=3, padding=1), # 16 x 14 x 14
            nn.ReLU(),
                        
            nn.Conv2d(16, 16, kernel_size=3, padding=1), # 16 x 14 x 14
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # 16 x 7 x 7
            
            nn.Conv2d(16, 32, kernel_size=3, padding=1), # 32 x 7 x 7
            nn.ReLU(),
                        
            nn.Conv2d(32, 32, kernel_size=3, padding=1), # 32 x 7 x 7
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # 32 x 3 x 3
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(32*3*3, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.convo_layers(x)
        x = x.view(x.size(0), -1) # flatten to 1d vector of 32 x 3 x 3
        return self.fc_layers(x)

In [7]:
cnn_model = CNN()

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(cnn_model.parameters())

## Training

In [8]:
epochs = 10
min_val_loss = float("inf")

for epoch in range(0, epochs):
    # training
    cnn_model.train()
    train_loss = 0.0

    for images, labels in train_loader:
        optimizer.zero_grad()
        y_pred = cnn_model.forward(images)
        loss = criterion(y_pred, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # validation
    cnn_model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for images, labels in val_loader:
            y_pred = cnn_model.forward(images)
            loss = criterion(y_pred, labels)
            val_loss += loss.item()

    if val_loss < min_val_loss:
        min_val_loss = val_loss
        torch.save(cnn_model.state_dict(), "cnn_digit_classifier.pt")

    print(f"Epoch: {epoch} Train loss => {train_loss / len(train_loader)} Val loss => {val_loss / len(val_loader)}")



Epoch: 0 Train loss => 0.5821568988561631 Val loss => 0.20465771798440752
Epoch: 1 Train loss => 0.17733070020377636 Val loss => 0.130404589559011
Epoch: 2 Train loss => 0.12164565891524155 Val loss => 0.12441040206938347
Epoch: 3 Train loss => 0.09850963121217986 Val loss => 0.0983794402737329
Epoch: 4 Train loss => 0.08292881548032165 Val loss => 0.11254081311338443
Epoch: 5 Train loss => 0.07542083723408481 Val loss => 0.07927860063005318
Epoch: 6 Train loss => 0.06864477299991995 Val loss => 0.07547099044932885
Epoch: 7 Train loss => 0.06612453836202621 Val loss => 0.09217631141902839
Epoch: 8 Train loss => 0.06257661876951655 Val loss => 0.06464450459486429
Epoch: 9 Train loss => 0.05794567887065932 Val loss => 0.06921556664809109


## Evaluation

In [9]:
cnn_model.load_state_dict(torch.load("cnn_digit_classifier.pt"))

C:\Users\User\AppData\Local\Temp\ipykernel_22776\3136356832.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cnn_model.load_state_dict(torch.load("cnn_digit_classifier.pt

<All keys matched successfully>

In [20]:
cnn_model.eval()

y_pred = []
y_actual = []

with torch.no_grad():
    for images, labels in test_loader:
        y_pred_logits = cnn_model(images)
        _, y_pred_labels = torch.max(y_pred_logits, 1)

        y_actual.extend(labels.cpu().numpy())
        y_pred.extend(y_pred_labels.cpu().numpy())

In [21]:
print(f"Accuracy: {accuracy_score(y_actual, y_pred)}")
print(classification_report(y_actual, y_pred))

Accuracy: 0.9819
              precision    recall  f1-score   support

           0       0.97      1.00      0.99       980
           1       0.99      1.00      0.99      1135
           2       0.95      0.98      0.97      1032
           3       0.97      1.00      0.98      1010
           4       0.99      0.99      0.99       982
           5       0.98      0.95      0.96       892
           6       0.99      0.96      0.98       958
           7       0.99      0.97      0.98      1028
           8       0.99      0.98      0.99       974
           9       0.99      0.98      0.98      1009

    accuracy                           0.98     10000
   macro avg       0.98      0.98      0.98     10000
weighted avg       0.98      0.98      0.98     10000

